In [ ]:

!cd Llava1.6/

In [ ]:
!ls


drive  Llava1.6  sample_data


In [ ]:
!git clone https://github.com/haotian-liu/LLaVA.git
!cd LLaVA

Cloning into 'LLaVA'...
remote: Enumerating objects: 2297, done.
remote: Total 2297 (delta 0), reused 0 (delta 0), pack-reused 2297 (from 1)
Receiving objects: 100% (2297/2297), 13.71 MiB | 31.34 MiB/s, done.
Resolving deltas: 100% (1404/1404), done.


In [ ]:
!nvidia-smi


Sat Jan 31 02:54:18 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   57C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install -e .
!pip install transformers accelerate sentencepiece bitsandbytes

Obtaining file:///content
ERROR: file:///content does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 14.3 MB/s eta 0:00:00


In [ ]:
!ls


cog.yaml  images   llava	   playground  pyproject.toml  scripts
docs	  LICENSE  llava.egg-info  predict.py  README.md


In [ ]:
!pip install transformers==4.36.2 accelerate==0.24.1 bitsandbytes
!git clone https://github.com/haotian-liu/LLaVA.git
%cd LLaVA
!pip install -e . --no-deps


fatal: destination path 'LLaVA' already exists and is not an empty directory.
/content/LLaVA
Obtaining file:///content/LLaVA
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for llava (pyproject.toml) ... done
  Created wheel for llava: filename=llava-1.2.2.post1-0.editable-py3-none-any.whl size=17891 sha256=32aed8578bbfdb25d7695e9ad06b43fe1cbc7ac4f69903ecd67c01334cfcbded
  Stored in directory: /tmp/pip-ephem-wheel-cache-9ymqqsbv/wheels/e5/0e/a5/e0f37963befcb0a4cf2bd7abeef83f743ef7d9ba56df1b852e
Successfully built llava
  Attempting uninstall: llava
    Found existing installation: llava 1.2.2.post1
    Uninstalling llava-1.2.2.post1:
      Successfully uninstalled llava-1.2.2.post1


In [ ]:
!pip install -e . --no-deps


Obtaining file:///content/LLaVA
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for llava (pyproject.toml) ... done
  Created wheel for llava: filename=llava-1.2.2.post1-0.editable-py3-none-any.whl size=17891 sha256=0a5327589c83fc83650bbcf4eb42803136478ed58ffff4caafe4251c00e264f8
  Stored in directory: /tmp/pip-ephem-wheel-cache-ziyky19q/wheels/e5/0e/a5/e0f37963befcb0a4cf2bd7abeef83f743ef7d9ba56df1b852e
Successfully built llava


In [ ]:
!pip install bitsandbytes sentencepiece


In [ ]:
from llava.model.builder import load_pretrained_model
from llava.mm_utils import get_model_name_from_path

model_path = "liuhaotian/llava-v1.6-vicuna-7b"
model_name = get_model_name_from_path(model_path)

tokenizer, model, image_processor, context_len = load_pretrained_model(
    model_path=model_path,
    model_base=None,
    model_name=model_name,
    device="cuda",
    load_4bit=True   # ⭐ THIS FIXES OOM
)

model.eval()
print("Loaded in 4-bit")



/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `for

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Loaded in 4-bit


In [ ]:
import os
import json
import torch
from PIL import Image


In [ ]:
from PIL import Image
from llava.constants import DEFAULT_IMAGE_TOKEN, IMAGE_TOKEN_INDEX
from llava.conversation import conv_templates
from llava.mm_utils import tokenizer_image_token, process_images
import torch

image = Image.open("test.jpg").convert("RGB")

conv = conv_templates["llava_v1"].copy()
conv.append_message(conv.roles[0], DEFAULT_IMAGE_TOKEN + "\nDescribe this image in detail.")
conv.append_message(conv.roles[1], None)

prompt = conv.get_prompt()

input_ids = tokenizer_image_token(
    prompt,
    tokenizer,
    IMAGE_TOKEN_INDEX,
    return_tensors="pt"
).unsqueeze(0).cuda()

image_tensor = process_images(
    [image],
    image_processor,
    model.config
).to(dtype=torch.float16, device="cuda")

image_sizes = [image.size]

with torch.no_grad():
    output_ids = model.generate(
        input_ids,
        images=image_tensor,
        image_sizes=image_sizes,
        max_new_tokens=128
    )

print(tokenizer.decode(output_ids[0], skip_special_tokens=True))


The image captures a scene of urban decay. A discarded orange and blue object, possibly a piece of clothing or a bag, lies on the ground next to a white brick wall. The wall, showing signs of age and weathering, is the only structure in the frame. The ground beneath the object is a patchwork of dirt and grass, hinting at the neglect of the area. The colors in the image are predominantly blue, orange, and white, creating a stark contrast against the gray of the ground. The object's vibrant colors stand out against the muted tones of the surr


In [ ]:
def llava_caption(image, prompt):
    conv = conv_templates["llava_v1"].copy()
    conv.append_message(conv.roles[0], DEFAULT_IMAGE_TOKEN + "\n" + prompt)
    conv.append_message(conv.roles[1], None)

    input_ids = tokenizer_image_token(
        conv.get_prompt(),
        tokenizer,
        IMAGE_TOKEN_INDEX,
        return_tensors="pt"
    ).unsqueeze(0).cuda()

    image_tensor = process_images(
        [image], image_processor, model.config
    ).to(dtype=torch.float16, device="cuda")

    with torch.no_grad():
        out = model.generate(
            input_ids,
            images=image_tensor,
            image_sizes=[image.size],
            max_new_tokens=80
        )

    return tokenizer.decode(out[0], skip_special_tokens=True)


In [ ]:
def llava_text_fusion(text):

    conv = conv_templates["llava_v1"].copy()
    conv.append_message(conv.roles[0], text)
    conv.append_message(conv.roles[1], None)

    input_ids = tokenizer(
        conv.get_prompt(),
        return_tensors="pt"
    ).input_ids.cuda()

    with torch.no_grad():
        out = model.generate(
            input_ids,
            max_new_tokens=120
        )

    return tokenizer.decode(out[0], skip_special_tokens=True)


In [ ]:
from PIL import Image

def dual_fusion(crop_path, full_path):

    crop_caption = llava_caption(
        Image.open(crop_path).convert("RGB"),
        "Describe the litter object precisely. Material, type, and size."
    )

    context_caption = llava_caption(
        Image.open(full_path).convert("RGB"),
        "Describe the surroundings and likely source of the litter."
    )

    fusion_prompt = f"""
You are analysing litter.

CROP ANALYSIS:
{crop_caption}

CONTEXT ANALYSIS:
{context_caption}

Combine both into ONE final explanation describing:
1. what the object is
2. its material
3. likely source
4. confidence

Be concise.
"""

    return llava_text_fusion(fusion_prompt)


In [ ]:
result = dual_fusion(
    "crops/crop_11.jpg",
    "images_full/img_11.jpg"
)

print(result)


The object in the image is a small, crumpled piece of paper, possibly a wrapper from a snack or drink, with the word "TREET" in a stylized font. The wrapper is predominantly orange with a blue border and is lying on the ground next to a blue brick wall. The material of the wrapper appears to be paper or cardboard. The likely source of the litter is a nearby establishment that discarded it irresponsibly. The confidence in this analysis is moderate, as the wrapper's label is not clearly visible, and the


In [ ]:
import json

def dual_fusion_structured(idx, crop_path, full_path):

    crop_img = Image.open(crop_path).convert("RGB")
    full_img = Image.open(full_path).convert("RGB")

    crop_text = llava_caption(
        crop_img,
        "Describe only the litter object precisely."
    )

    context_text = llava_caption(
        full_img,
        "Describe the surrounding environment and litter distribution."
    )

    fusion_prompt = f"""
You are analysing litter for an environmental dataset.

CROP DESCRIPTION:
{crop_text}

CONTEXT DESCRIPTION:
{context_text}

Return ONLY valid JSON:

{{
  "perception_score": 0,
  "dominant_litter_source": "",
  "visual_density": "",
  "spatial_distribution": "",
  "confidence": 0.0
}}
"""

    fusion_text = llava_text_fusion(fusion_prompt)

    try:
        model_output = json.loads(fusion_text)
    except:
        model_output = {"raw_output": fusion_text}

    return {
        "id": idx,
        "crop_image": crop_path,
        "context_image": full_path,
        "crop_text": crop_text,
        "context_text": context_text,
        "model_output": model_output
    }


In [ ]:
results = []

for crop_file in sorted(os.listdir("crops")):

    idx = crop_file.split("_")[1].split(".")[0]

    result = dual_fusion_structured(
        idx,
        f"crops/crop_{idx}.jpg",
        f"images_full/img_{idx}.jpg"
    )

    results.append(result)

with open("results_llava.json", "w") as f:
    json.dump(results, f, indent=2)


In [ ]:
!mv /content/LLaVA /content/drive/MyDrive/llava_project/


In [1]:
!git config --global user.email "anjyibitoye@gmail.com"
!git config --global user.name "AnjolaIbitoye"


In [3]:
%cd /content/drive/MyDrive
!git clone https://github.com/AnjolaIbitoye/litter-vlm-project.git
%cd litter-vlm-project


/content/drive/MyDrive
Cloning into 'litter-vlm-project'...
remote: Enumerating objects: 3, done.
remote: Counting objects: 100% (3/3), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 3 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (3/3), done.
/content/drive/MyDrive/litter-vlm-project


In [5]:
%cd /content/drive/MyDrive/llava_project


/content/drive/MyDrive/llava_project


In [6]:
!ls


LLaVA  Llava_notebook.ipynb


In [10]:
!git init



hint: Using 'master' as the name for the initial branch. This default branch name
hint: is subject to change. To configure the initial branch name to use in all
hint: of your new repositories, which will suppress this warning, call:
hint: 
hint: 	git config --global init.defaultBranch <name>
hint: 
hint: Names commonly chosen instead of 'master' are 'main', 'trunk' and
hint: 'development'. The just-created branch can be renamed via this command:
hint: 
hint: 	git branch -m <name>
Initialized empty Git repository in /content/drive/MyDrive/llava_project/.git/


In [11]:
!git remote add origin https://github.com/AnjolaIbitoye/litter-vlm-project.git
!git add .
!git commit -m "Initial commit"
!git branch -M main
!git push -u origin main


hint: You've added another git repository inside your current repository.
hint: Clones of the outer repository will not contain the contents of
hint: the embedded repository and will not know how to obtain it.
hint: If you meant to add a submodule, use:
hint: 
hint: 	git submodule add <url> LLaVA
hint: 
hint: If you added this path by mistake, you can remove it from the
hint: index with:
hint: 
hint: 	git rm --cached LLaVA
hint: 
hint: See "git help submodule" for more information.
[master (root-commit) ba7cec3] Initial commit
 2 files changed, 2 insertions(+)
 create mode 160000 LLaVA
 create mode 100644 Llava_notebook.ipynb
fatal: could not read Username for 'https://github.com': No such device or address


In [12]:
!git rm --cached -r LLaVA


rm 'LLaVA'
